In [7]:
import numpy as np
############################################################
# LOAD PREPROCESSED DATA
############################################################

import pandas as pd

# Load merged dataset created during preprocessing
df = pd.read_csv("merged_data.csv")

# Convert TransactionDate back to datetime
df["TransactionDate"] = pd.to_datetime(
    df["TransactionDate"],
    errors="coerce"
)

print("Merged data loaded successfully.")
print(df.head())


class FeatureEngineer:

    def __init__(self, data):

        self.data = data.copy()
        self.profile = None

    #######################################################
    # Total Courses Enrolled
    #######################################################

    def total_courses(self):

        total = self.data.groupby("UserID")["CourseID"].count()

        return total

    #######################################################
    # Total Amount Spent
    #######################################################

    def total_spent(self):

        spent = self.data.groupby("UserID")["Amount"].sum()

        return spent

    #######################################################
    # Average Spending
    #######################################################

    def average_spending(self):

        avg = self.data.groupby("UserID")["Amount"].mean()

        return avg

    #######################################################
    # Preferred Category
    #######################################################

    def preferred_category(self):

        category = self.data.groupby("UserID")["CourseCategory"] \
            .agg(lambda x: x.mode().iloc[0])

        return category

    #######################################################
    # Preferred Level
    #######################################################

    def preferred_level(self):

        level = self.data.groupby("UserID")["CourseLevel"] \
            .agg(lambda x: x.mode().iloc[0])

        return level

    #######################################################
    # Average Rating
    #######################################################

    def average_rating(self):

        rating = self.data.groupby("UserID")["CourseRating"].mean()

        return rating

    #######################################################
    # Diversity Score
    #######################################################

    def diversity_score(self):

        diversity = self.data.groupby("UserID")["CourseCategory"].nunique()

        return diversity

    #######################################################
    # Enrollment Frequency
    #######################################################

    def enrollment_frequency(self):

        temp = self.data.copy()

        temp["Month"] = temp["TransactionDate"].dt.to_period("M")

        frequency = temp.groupby("UserID").agg(

            TotalCourses=("CourseID", "count"),

            ActiveMonths=("Month", "nunique")

        )

        frequency["EnrollmentFrequency"] = (

            frequency["TotalCourses"]

            /

            frequency["ActiveMonths"]

        )

        return frequency["EnrollmentFrequency"]

    #######################################################
    # Learning Depth Index
    #######################################################

    def learning_depth(self):

        temp = self.data.copy()

        beginner = temp[temp["CourseLevel"] == "Beginner"] \
            .groupby("UserID")["CourseID"].count()

        intermediate = temp[temp["CourseLevel"] == "Intermediate"] \
            .groupby("UserID")["CourseID"].count()

        advanced = temp[temp["CourseLevel"] == "Advanced"] \
            .groupby("UserID")["CourseID"].count()

        users = self.data["UserID"].unique()

        beginner = beginner.reindex(users, fill_value=0)

        intermediate = intermediate.reindex(users, fill_value=0)

        advanced = advanced.reindex(users, fill_value=0)

        depth = (

            advanced * 2 +

            intermediate

        ) / (

            beginner + 1

        )

        depth.index = users

        return depth

    #######################################################
    # Average Courses Per Category
    #######################################################

    def average_courses_per_category(self):

        total = self.total_courses()

        diversity = self.diversity_score()

        average = total / diversity

        return average

    #######################################################
    # Most Preferred Payment Method
    #######################################################

    def payment_method(self):

        payment = self.data.groupby("UserID")["PaymentMethod"] \
            .agg(lambda x: x.mode().iloc[0])

        return payment

    #######################################################
    # Average Course Price
    #######################################################

    def average_course_price(self):

        price = self.data.groupby("UserID")["CoursePrice"].mean()

        return price

        #######################################################
    # Create Learner Profile
    #######################################################

    def create_profile(self):

        print("Creating learner profile...")

        profile = pd.DataFrame(index=self.data["UserID"].unique())

        # Feature Engineering
        profile["TotalCourses"] = self.total_courses()

        profile["TotalSpent"] = self.total_spent()

        profile["AverageSpending"] = self.average_spending()

        profile["PreferredCategory"] = self.preferred_category()

        profile["PreferredLevel"] = self.preferred_level()

        profile["AverageCourseRating"] = self.average_rating()

        profile["DiversityScore"] = self.diversity_score()

        profile["EnrollmentFrequency"] = self.enrollment_frequency()

        profile["LearningDepthIndex"] = self.learning_depth()

        profile["AverageCoursesPerCategory"] = (
            self.average_courses_per_category()
        )

        profile["PreferredPaymentMethod"] = self.payment_method()

        profile["AverageCoursePrice"] = self.average_course_price()

        ###################################################
        # User Demographics
        ###################################################

        user_info = self.data[
            ["UserID", "Age", "Gender"]
        ].drop_duplicates()

        user_info = user_info.set_index("UserID")

        profile = profile.join(user_info)

        ###################################################
        # Fill Missing Values
        ###################################################

        numeric_columns = profile.select_dtypes(include=np.number).columns

        for col in numeric_columns:

            profile[col] = profile[col].fillna(
                profile[col].median()
            )

        categorical_columns = profile.select_dtypes(
            include="object"
        ).columns

        for col in categorical_columns:

            profile[col] = profile[col].fillna(
                profile[col].mode()[0]
            )

        ###################################################
        # Reset Index
        ###################################################

        profile.reset_index(inplace=True)

        profile.rename(
            columns={"index": "UserID"},
            inplace=True
        )

        self.profile = profile

        print("Learner Profile Created Successfully")

        print(profile.head())

        return profile

    #######################################################
    # Save Learner Profile
    #######################################################

    def save(self, filename="learner_profiles.csv"):

        if self.profile is None:

            raise ValueError(
                "Run create_profile() first."
            )

        self.profile.to_csv(filename, index=False)

        print(f"{filename} saved successfully.")

Merged data loaded successfully.
  TransactionID  UserID CourseID TransactionDate  Amount  PaymentMethod  \
0       TT00001  U00003  CR00016      2025-10-25     0.0         PayPal   
1       TT00002  U00003  CR00037      2025-01-13     0.0         PayPal   
2       TT00003  U00003  CR00019      2025-03-28     0.0  Bank Transfer   
3       TT00004  U00004  CR00048      2025-06-02     0.0  Bank Transfer   
4       TT00005  U00004  CR00060      2025-08-10     0.0         PayPal   

  TeacherID         CourseName           CourseCategory CourseType  \
0   TC00040  Digital Marketing                Marketing       Free   
1   TC00040   Scrum Essentials       Project Management       Free   
2   TC00040  Content Marketing                Marketing       Free   
3   TC00040          Ai Ethics  Artificial Intelligence       Free   
4   TC00042   Content Creation        Digital Marketing       Free   

    CourseLevel  CoursePrice  CourseDuration  CourseRating        UserName  \
0  Intermediate  

In [8]:
# df should already exist from preprocessing.ipynb

fe = FeatureEngineer(df)

learner_profile = fe.create_profile()

fe.save()

learner_profile.head()

Creating learner profile...
Learner Profile Created Successfully
   UserID  TotalCourses  TotalSpent  AverageSpending        PreferredCategory  \
0  U00003            11      613.98        55.816364                Marketing   
1  U00004            13      982.05        75.542308        Digital Marketing   
2  U00013            14     1432.33       102.309286         Machine Learning   
3  U00015            14     2035.06       145.361429  Artificial Intelligence   
4  U00018            16     1468.03        91.751875  Artificial Intelligence   

  PreferredLevel  AverageCourseRating  DiversityScore  EnrollmentFrequency  \
0       Advanced             2.889091               7             1.571429   
1   Intermediate             3.451538               8             1.625000   
2       Advanced             2.869286               8             1.555556   
3       Beginner             2.760714              10             2.000000   
4       Beginner             3.108125               9     

,UserID,TotalCourses,TotalSpent,AverageSpending,PreferredCategory,PreferredLevel,AverageCourseRating,DiversityScore,EnrollmentFrequency,LearningDepthIndex,AverageCoursesPerCategory,PreferredPaymentMethod,AverageCoursePrice,Age,Gender
0,U00003,11,613.98,55.816364,Marketing,Advanced,2.889091,7,1.571429,2.200000,1.571429,Bank Transfer,55.816364,33,Female
1,U00004,13,982.05,75.542308,Digital Marketing,Intermediate,3.451538,8,1.625000,2.400000,1.625000,Bank Transfer,75.542308,23,Female
2,U00013,14,1432.33,102.309286,Machine Learning,Advanced,2.869286,8,1.555556,2.333333,1.750000,Bank Transfer,102.309286,33,Female
3,U00015,14,2035.06,145.361429,Artificial Intelligence,Beginner,2.760714,10,2.000000,1.250000,1.400000,Bank Transfer,145.361429,27,Female
4,U00018,16,1468.03,91.751875,Artificial Intelligence,Beginner,3.108125,9,1.454545,1.625000,1.777778,Credit Card,91.751875,24,Female
